# Crescendo — Spike Report (v0.1)

**Headline question:** *Is emerging-artist breakout predictable from momentum features on self-collected YouTube data, proven with a leakage-safe eval, before building any game?*

This notebook imports the `crescendo` package and renders the answer: precision@k vs base-rate and naive-momentum baselines on a **temporal, per-fold** holdout. It reads the same `dataset` table the CLI builds — nothing is recomputed here that isn't in the pipeline.

> Runs meaningfully only after ~45 days of collected history (or a backfill). Until then it renders on whatever `dataset` rows exist.

In [ ]:
from crescendo.config import load_config
from crescendo.db import Db
from crescendo.dataset import dataset_version
from crescendo.evaluate import evaluate

cfg = load_config()
db = Db(cfg.database_url)
df = db.read_dataset(version=dataset_version(cfg))
print(f'dataset rows: {len(df)}  version: {dataset_version(cfg)}')
df.head()

In [ ]:
# Walk-forward evaluation: model vs baselines, precision@k and lift per fold.
results = evaluate(cfg, db, cutoff=cfg.cutoff, k='auto', walk_forward=True)
for r in results:
    print(f'fold {r.fold_index} @ {r.cutoff}: P@{r.k}={r.precision_at_k:.3f} '
          f'base={r.base_rate:.3f} lift={r.lift:.2f} auc={r.roc_auc:.3f}')

## Verdict

- **lift > 1** across folds → momentum features carry real breakout signal (positive result).
- **lift ≈ 1** → no edge over base rate; an honest negative result is still a valid resume story (per L1 §8).

The `reasons` (feature importances) from `model.feature_importances()` feed the transparent-AI opponent in v1.1.